Model Training & Evaluation
ในขั้นตอนนี้ เราจะสร้างไฟล์ Jupyter Notebook ที่สอง (เช่น 02_model_experiment.ipynb) เพื่อทำการแข่งขันหาโมเดลที่แม่นยำที่สุด โดยผมจะวางโครงสร้างโค้ดตามมาตรฐานที่อาจารย์ของคุณน่าจะคาดหวัง ดังนี้ครับ:

Data Preprocessing:

ทำ One-Hot Encoding แปลงข้อมูลข้อความ (เช่น อาชีพ, ประเภทสินเชื่อ) ให้เป็นเลข 0 กับ 1

ทำ StandardScaler ปรับสเกลตัวเลข (เช่น อายุ, รายได้, วงเงินกู้) ให้อยู่ในมาตรฐานเดียวกัน เพื่อให้ Logistic Regression และ SVM ทำงานได้เต็มประสิทธิภาพ

Train-Test Split: แบ่งข้อมูลเป็นชุดเรียนรู้ (Train 80%) และชุดสอบ (Test 20%)

SMOTE: นำเทคนิคแก้ไขข้อมูลไม่สมดุลมาใช้ เฉพาะในชุด Train 80% เพื่อป้องกันปัญหา Data Leakage

Model Training & Comparison: เทรนพร้อมกัน 4 อัลกอริทึม (Logistic Regression, Random Forest, XGBoost, SVM)

Evaluation: สรุปผลการแข่งขันออกมาเป็นตารางเปรียบเทียบค่า F1-Score และ ROC-AUC

*โหลดข้อมูลและสร้างระบบจัดการข้อมูล (Data Preprocessing)*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
# 1. โหลดข้อมูล
df = pd.read_csv('../data/raw/islamic_coop_synthetic_data_v2.csv')

In [3]:
# 2. แยกตัวแปรต้น (X) และตัวแปรตาม (y)
X = df.drop('Target_NPF', axis=1)
y = df['Target_NPF']

In [4]:
# 3. คัดแยกประเภทคอลัมน์
categorical_cols = ['Occupation', 'Financing_Type', 'Guarantor_Type']
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [5]:
# 4. สร้าง Preprocessor (แปลงตัวเลขให้อยู่ในสเกลเดียวกัน และแปลงข้อความเป็นเลข 0/1)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

In [6]:
# 5. แบ่งข้อมูล Train 80% / Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"จำนวนชุด Train (สำหรับให้ AI เรียนรู้): {X_train.shape[0]} แถว")
print(f"จำนวนชุด Test (สำหรับสอบวัดผล): {X_test.shape[0]} แถว")

จำนวนชุด Train (สำหรับให้ AI เรียนรู้): 8000 แถว
จำนวนชุด Test (สำหรับสอบวัดผล): 2000 แถว


*แข่งขัน 4 อัลกอริทึม (Model Training & Evaluation)*
-จุดเด่นของโค้ดส่วนนี้คือการใช้ ImbPipeline ซึ่งจะสั่งให้ SMOTE ปั๊มข้อมูลหนี้เสียเฉพาะในชุด Train เท่านั้น ป้องกันปัญหาข้อมูลรั่วไหล (Data Leakage)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

In [8]:
# 6. เตรียมตัวผู้เข้าแข่งขันทั้ง 4 โมเดล
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM': SVC(probability=True, random_state=42)
}

results = []
best_f1 = 0
best_model_name = ""
best_pipeline = None

In [9]:
# 7. ปล่อยโมเดลวิ่งทีละตัว
for name, model in models.items():
    # มัดรวมขั้นตอน: ทำความสะอาดข้อมูล -> ปั๊มข้อมูลให้สมดุล -> เทรนโมเดล
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    
    # สอบเข้าเรียน (Train)
    pipeline.fit(X_train, y_train)
    
    # สอบปลายภาค (Test - SMOTE จะไม่ทำงานในขั้นตอนนี้)
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # วัดคะแนน
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results.append({'Model': name, 'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': roc_auc})
    
    # คัดเลือกโมเดลที่ดีที่สุดจาก F1-Score
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_pipeline = pipeline

In [10]:
# 8. สรุปผลการแข่งขันเรียงตามคะแนน F1-Score
results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
display(results_df)

,Model,Accuracy,F1-Score,ROC-AUC
0,Logistic Regression,0.6515,0.516979,0.687381
1,SVM,0.6450,0.503497,0.671864
2,Random Forest,0.6790,0.404453,0.680889
3,XGBoost,0.6850,0.397706,0.660850


*ส่งออกโมเดลผู้ชนะ (Export Best Model)*

In [11]:
import joblib
import os

In [12]:
# 9. บันทึกโมเดลและ Preprocessor แบบมัดรวมแพ็กเกจ
model_dir = '../models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

model_path = os.path.join(model_dir, 'best_credit_model.pkl')

In [13]:
# บันทึกเป็นไฟล์ .pkl (Pickle)
joblib.dump(best_pipeline, model_path)

print(f"🏆 โมเดลผู้ชนะคือ: '{best_model_name}'")
print(f"💾 บันทึกไฟล์เรียบร้อยแล้วที่: {model_path}")
print("🚀 พร้อมนำไปเสียบใช้งานใน Streamlit Web App แล้ว!")

🏆 โมเดลผู้ชนะคือ: 'Logistic Regression'
💾 บันทึกไฟล์เรียบร้อยแล้วที่: ../models\best_credit_model.pkl
🚀 พร้อมนำไปเสียบใช้งานใน Streamlit Web App แล้ว!


GridSearchCV (ค้นหาแบบปูพรมทุกซอกทุกมุม)

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve, f1_score, recall_score
import joblib
import os
import numpy as np

print("⏳ เริ่มกระบวนการค้นหา Parameter ที่ดีที่สุดแบบปูพรม (GridSearchCV)...")
print("(อาจใช้เวลา 5-15 นาที ขึ้นอยู่กับความเร็วคอมพิวเตอร์ของคุณ กรุณารอจนกว่าจะเสร็จ)")

# ==================================================
# ส่วนที่ 1: GridSearchCV ป้องกัน Overfitting
# ==================================================
# 1. กำหนดขอบเขตการค้นหา Parameter สำหรับ Random Forest
# จำกัดความลึกของต้นไม้ (max_depth) ไม่ให้ลึกเกินไปจนจำข้อสอบ
rf_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10], 
    'classifier__min_samples_split': [2, 5]
}

# สร้างท่อ (Pipeline) ใหม่สำหรับ Random Forest (มี SMOTE ในตัว)
rf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# สั่งให้ระบบค้นหา (ให้คะแนนการสอบด้วย F1-Score เป็นหลัก)
grid_rf = GridSearchCV(rf_pipeline, rf_params, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)

# 2. เปรียบเทียบผู้ชนะคนใหม่กับ Logistic Regression ตัวเดิม
best_new_model = grid_rf.best_estimator_
y_pred_new = best_new_model.predict(X_test)
y_pred_proba_new = best_new_model.predict_proba(X_test)[:, 1]

new_f1 = f1_score(y_test, y_pred_new)
print(f"🎯 F1-Score ของ Random Forest (หลังจูน): {new_f1:.4f}")
print(f"🌲 Parameters ที่ดีที่สุด: {grid_rf.best_params_}")

# เลือกโมเดลที่ F1 สูงสุด (เทียบแชมป์เก่า vs ผู้ท้าชิงใหม่)
if new_f1 > best_f1:
    print("🏆 Random Forest แย่งแชมป์สำเร็จ! เราจะใช้โมเดลนี้เป็นตัวหลัก")
    final_model = best_new_model
else:
    print("🏆 Logistic Regression ตัวเดิมยังคงป้องกันแชมป์ไว้ได้!")
    final_model = best_pipeline

# ==================================================
# ส่วนที่ 2: Threshold Moving ขยับเกณฑ์ให้เน้นจับหนี้เสีย
# ==================================================
print("\n--- เริ่มกระบวนการหาจุดตัดที่คุ้มค่าที่สุด (Optimal Threshold) ---")

# หาค่าความน่าจะเป็นจากชุดทดสอบ (Test Set) ด้วยโมเดลผู้ชนะ
y_scores = final_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

# หาจุดที่ให้ค่า F1-Score สูงสุดบน Threshold ใหม่
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
# ป้องกันค่าหารด้วย 0
f1_scores = np.nan_to_num(f1_scores)

best_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_idx]
optimal_f1 = f1_scores[best_idx]

print(f"⚖️ เกณฑ์ความเสี่ยง (Threshold) เดิมคือ 0.5")
print(f"🔍 เกณฑ์ความเสี่ยงใหม่ที่ดีที่สุด (Optimal Threshold) คือ: {optimal_threshold:.4f}")
print(f"📈 F1-Score หากใช้เกณฑ์ใหม่นี้จะพุ่งไปที่: {optimal_f1:.4f}")

# ทดลองทำนายผลลัพธ์ด้วยเกณฑ์ใหม่
y_pred_optimal = (y_scores >= optimal_threshold).astype(int)
print(f"✅ ความสามารถใหม่: หนี้เสีย 100 คน โมเดลจับได้ (Recall) ถึง {recall_score(y_test, y_pred_optimal)*100:.1f} คน")

# ==================================================
# ส่วนที่ 3: อัปเดตไฟล์โมเดล
# ==================================================
# บันทึกโมเดลตัวสุดท้ายพร้อม Threshold ค่าใหม่ทับไฟล์เดิม
final_package = {
    'pipeline': final_model,
    'optimal_threshold': optimal_threshold
}

joblib.dump(final_package, model_path)
print(f"\n💾 บันทึกโมเดลและ Threshold ใหม่ทับไฟล์เดิมเรียบร้อยแล้ว!")

⏳ เริ่มกระบวนการค้นหา Parameter ที่ดีที่สุดแบบปูพรม (GridSearchCV)...
(อาจใช้เวลา 5-15 นาที ขึ้นอยู่กับความเร็วคอมพิวเตอร์ของคุณ กรุณารอจนกว่าจะเสร็จ)
Fitting 3 folds for each of 8 candidates, totalling 24 fits
🎯 F1-Score ของ Random Forest (หลังจูน): 0.5288
🌲 Parameters ที่ดีที่สุด: {'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
🏆 Random Forest แย่งแชมป์สำเร็จ! เราจะใช้โมเดลนี้เป็นตัวหลัก

--- เริ่มกระบวนการหาจุดตัดที่คุ้มค่าที่สุด (Optimal Threshold) ---
⚖️ เกณฑ์ความเสี่ยง (Threshold) เดิมคือ 0.5
🔍 เกณฑ์ความเสี่ยงใหม่ที่ดีที่สุด (Optimal Threshold) คือ: 0.4211
📈 F1-Score หากใช้เกณฑ์ใหม่นี้จะพุ่งไปที่: 0.5608
✅ ความสามารถใหม่: หนี้เสีย 100 คน โมเดลจับได้ (Recall) ถึง 82.6 คน

💾 บันทึกโมเดลและ Threshold ใหม่ทับไฟล์เดิมเรียบร้อยแล้ว!
